<a href="https://colab.research.google.com/github/minxuan0923/AML_Assignment/blob/mx/Set5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import libraries

In [23]:
import pandas as pd
from scipy.io import arff
import requests
from io import StringIO

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

import tensorflow as tf
from tensorflow.keras import datasets, layers, models, optimizers, losses
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

## Load Dataset

In [34]:
URL = (
    "https://raw.githubusercontent.com/"
    "minxuan0923/AML_Assignment/main/data/"
    "UCI_Phishing_Training_Dataset.arff"
)

response = requests.get(URL, timeout=30)
response.raise_for_status()

data, meta = arff.loadarff(StringIO(response.text))

df = pd.DataFrame(data)

for col in df.select_dtypes([object]):
    df[col] = pd.to_numeric(df[col].str.decode('utf-8'))

X_data = df.drop(columns =['Result'])
y_data = df['Result']

## Spliting

In [35]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, random_state = 42, test_size=0.3, stratify=y_data)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


## Encoding

In [28]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

# One-hot-encoding X values
X_train_encoded = encoder.fit_transform(X_train)
X_val_encoded = encoder.fit_transform(X_val)
X_test_encoded = encoder.transform(X_test)

# Convert y values - phishing to "1" and legitimate to "0"
y_train = y_train.replace({-1: 1, 1: 0})
y_val = y_val.replace({-1: 1, 1: 0})
y_test = y_test.replace({-1: 1, 1: 0})

## Autoencoder

In [30]:

class Autoencoder(Model):
  def __init__(self, latent_dim, shape):
    super(Autoencoder, self).__init__()
    self.latent_dim = latent_dim
    self.shape = shape
    self.encoder = tf.keras.Sequential([
      layers.Flatten(),
      layers.Dense(latent_dim, activation='relu'),
    ])
    self.decoder = tf.keras.Sequential([
      layers.Dense(int(tf.math.reduce_prod(shape)), activation='sigmoid'),
      layers.Reshape(shape)
    ])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded


shape = X_test_encoded.shape[1:]
latent_dim = 16 # we will experiment with 4, 8, 16, 32
autoencoder = Autoencoder(latent_dim, shape)

autoencoder.compile(optimizer='adam', loss=losses.binary_crossentropy)

history_ae = autoencoder.fit(X_train_encoded, X_train_encoded,
                epochs=10,
                shuffle=True,
                validation_data=(X_val_encoded, X_val_encoded))

X_train_compressed = autoencoder.encoder.predict(X_train_encoded)
X_val_compressed = autoencoder.encoder.predict(X_val_encoded)
X_test_compressed = autoencoder.encoder.predict(X_test_encoded)

Epoch 1/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.4693 - val_loss: 0.3569
Epoch 2/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.3063 - val_loss: 0.2650
Epoch 3/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.2360 - val_loss: 0.2063
Epoch 4/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1839 - val_loss: 0.1649
Epoch 5/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1497 - val_loss: 0.1373
Epoch 6/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1263 - val_loss: 0.1186
Epoch 7/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1107 - val_loss: 0.1066
Epoch 8/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1004 - val_loss: 0.0982
Epoch 9/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0922 - val_loss: 0.0891
Epoch 10/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0821 - val_loss: 0.0800
242/242 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


## Logistic Regression

In [31]:
# Gradient descent
logistic_model = models.Sequential()
logistic_model.add(layers.Normalization(input_shape = [16,], axis = None))
logistic_model.add(layers.Dense(1, activation = 'sigmoid'))

logistic_model.compile(optimizer='adam',
              loss=losses.binary_crossentropy,
              metrics=['accuracy'])

logistic_model.summary()

history_lr = logistic_model.fit(X_train_compressed, y_train, epochs=50,
                    validation_data=(X_val_compressed, y_val))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_2 (Normalization) │ (None, 16)             │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20 (84.00 B)

 Trainable params: 17 (68.00 B)

 Non-trainable params: 3 (16.00 B)

Epoch 1/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3590 - loss: 2.9087 - val_accuracy: 0.3878 - val_loss: 2.3865
Epoch 2/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4581 - loss: 1.8026 - val_accuracy: 0.5048 - val_loss: 1.4013
Epoch 3/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5916 - loss: 1.0140 - val_accuracy: 0.6520 - val_loss: 0.7711
Epoch 4/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7316 - loss: 0.5759 - val_accuracy: 0.7889 - val_loss: 0.4620
Epoch 5/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8302 - loss: 0.3773 - val_accuracy: 0.8625 - val_loss: 0.3339
Epoch 6/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8746 - loss: 0.2913 - val_accuracy: 0.8896 - val_loss: 0.2656
Epoch 7/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8971 - loss: 0.2481 - val_accuracy: 0.9125 - val_loss: 0.2333
Epoch 8/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9107 - loss: 0.2244 - val_accuracy: 0.

## Hyperparameter Tuning

In [33]:
test_loss, test_accuracy = logistic_model.evaluate(X_val_compressed, y_val)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9331 - loss: 0.1613
Test Accuracy: 0.9330518841743469
Test Loss: 0.16133326292037964


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history_lr.history['accuracy'], label='Training Accuracy')
plt.plot(history_lr.history['val_accuracy'], label='Validation Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()